# Loading data

In [8]:
import pandas as pd

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
with open('/content/drive/MyDrive/assignments/ITRex, men-women problem/training_data/female.txt') as f:
    lines = f.readlines()
    for line in lines[:10]:
        print(line)

they be wts 😂

Ill see you soon i gotta ger in the shower

No it's waiting for approval

Other times it’s quiet

oww okay thank you 😊

Well if u out this way tonight hit me up

Ohhhh will u so livw joy??

I was about to say u see wat I see🍆

😂😂😂😂😂😂😂😂😂😂😂😂 hat came from the Beauty SUPPLY

Cause you always in my DMs koko



In [11]:
def read_data(file_path, label):
    with open(file_path, 'r') as f:
        lines = [l.strip() for l in f.readlines()]
    df = pd.DataFrame({'text': lines, 'label': label})
    return df


female_df = read_data('/content/drive/MyDrive/assignments/ITRex, men-women problem/training_data/female.txt', 'female')
male_df = read_data('/content/drive/MyDrive/assignments/ITRex, men-women problem/training_data/male.txt', 'male')

combined_df = pd.concat([female_df, male_df], ignore_index=True)

combined_df

,text,label
0,they be wts 😂,female
1,Ill see you soon i gotta ger in the shower,female
2,No it's waiting for approval,female
3,Other times it’s quiet,female
4,oww okay thank you 😊,female
...,...,...
59995,THATs how it starts,male
59996,Yes circle of life,male
59997,When you comin to see me boo?! ❤️🍫,male
59998,The only thing i like to rub is some puss,male


In [12]:
combined_df.label.value_counts()

,count
label,
female,30000
male,30000


In [13]:
df = combined_df

In [14]:
df.text.str.len().describe()

,text
count,60000.000000
mean,28.015250
std,10.882625
min,9.000000
25%,21.000000
50%,25.000000
75%,33.000000
max,140.000000


In [15]:
df['class_name'] = df['label'].map({'female': 0, 'male': 1})
df.head()

,text,label,class_name
0,they be wts 😂,female,0
1,Ill see you soon i gotta ger in the shower,female,0
2,No it's waiting for approval,female,0
3,Other times it’s quiet,female,0
4,oww okay thank you 😊,female,0


In [16]:
df['class_name'], df['label'] = df['label'], df['class_name']
df.head()

,text,label,class_name
0,they be wts 😂,0,female
1,Ill see you soon i gotta ger in the shower,0,female
2,No it's waiting for approval,0,female
3,Other times it’s quiet,0,female
4,oww okay thank you 😊,0,female


In [25]:
df.dtypes

,0
text,object
label,int64
class_name,object


# Training model

## BERT

In [ ]:
!pip install datasets

In [38]:
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import numpy as np

In [39]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [40]:
df.dtypes

,0
text,object
label,int64
class_name,object


In [41]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, predictions),
            "precision": precision_score(labels, predictions),
            "recall": recall_score(labels, predictions),
            "accuracy": accuracy_score(labels, predictions),
            }


training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True,  # Enable mixed precision for speedup
    dataloader_num_workers=4,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

...

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-41-8494b473f0e1>:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker n

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,No log,0.574590,0.686515,0.720142,0.655889,0.700500
2,0.589500,0.550210,0.716045,0.731871,0.700889,0.722056
3,0.589500,0.569424,0.711932,0.753693,0.674556,0.727056
4,0.464300,0.595763,0.715991,0.748063,0.686556,0.727667


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: T

Evaluation Results: {'eval_loss': 0.5957626104354858, 'eval_f1': 0.7159907300115875, 'eval_precision': 0.7480629539951574, 'eval_recall': 0.6865555555555556, 'eval_accuracy': 0.7276666666666667, 'eval_runtime': 45.1315, 'eval_samples_per_second': 398.835, 'eval_steps_per_second': 49.854, 'epoch': 4.0}


In [42]:
# Evaluate the model
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Evaluation Results: {'eval_loss': 0.5957626104354858, 'eval_f1': 0.7159907300115875, 'eval_precision': 0.7480629539951574, 'eval_recall': 0.6865555555555556, 'eval_accuracy': 0.7276666666666667, 'eval_runtime': 50.742, 'eval_samples_per_second': 354.736, 'eval_steps_per_second': 44.342, 'epoch': 4.0}


## Catboost

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import numpy as np

In [18]:
# !pip install --upgrade numpy
# !pip install --force-reinstall catboost
# !pip install catboost

In [19]:
import catboost
from catboost import CatBoostClassifier, Pool

In [20]:
print(df.shape)
df.sample(10)

(60000, 3)


,text,label,class_name
4859,I’m in the bath lol,0,female
25192,Lol i love her because you always know when it...,0,female
45658,Yea I'm a photographer here,1,male
22459,Try tmw I will be live,0,female
3521,αll thαt nσíѕє lσl,0,female
35260,Lemme get in the box lol 😂,1,male
29085,Some time hit me up on here ..,0,female
31648,What it smells like though,1,male
24890,She want a gf lol :),0,female
56489,U have bigger toys,1,male


In [21]:
train_df, test_df = train_test_split(df, test_size=0.4, random_state=42, stratify=df["label"])
print(train_df.shape)
train_df.head()

(36000, 3)


,text,label,class_name
19407,Shout s mg family jn,0,female
8723,It complements your eyes! Wayyy better than blue.,0,female
42477,Nah I’m in the very beginning,1,male
3425,Kiss them cheek bones,0,female
21444,Lili I am a male but the app set me as a female,0,female


In [22]:
train_pool = Pool(data=train_df[['text']],  # Pass as DataFrame with double brackets
                  label=train_df["label"],
                  text_features=[0],  # Use column index 0 for 'text'
                  feature_names=['text'])

test_pool = Pool(data=test_df[['text']],
                 label=test_df["label"],
                 text_features=[0],
                 feature_names=['text'])

In [23]:
model = CatBoostClassifier(iterations=300,
                        #    depth=8,
                        #    task_type="GPU",
                        #    devices='0'
                           )

model.fit(train_pool, verbose=100)

Learning rate set to 0.143534
0:	learn: 0.6551312	total: 744ms	remaining: 3m 42s
100:	learn: 0.5329730	total: 38.8s	remaining: 1m 16s
200:	learn: 0.5155787	total: 1m 13s	remaining: 36s
299:	learn: 0.5020999	total: 1m 45s	remaining: 0us


In [24]:
from sklearn.metrics import accuracy_score, classification_report

predicted_class = model.predict(test_pool)
print(classification_report(test_df["label"], predicted_class))

              precision    recall  f1-score   support

           0       0.76      0.72      0.74     12000
           1       0.73      0.77      0.75     12000

    accuracy                           0.74     24000
   macro avg       0.75      0.74      0.74     24000
weighted avg       0.75      0.74      0.74     24000



# Testing script

In [71]:
df.head()

,text,label,class_name
0,they be wts 😂,0,female
1,Ill see you soon i gotta ger in the shower,0,female
2,No it's waiting for approval,0,female
3,Other times it’s quiet,0,female
4,oww okay thank you 😊,0,female


In [70]:
df.dtypes

,0
text,object
label,int64
class_name,object


In [ ]:
!pip install catboost

In [38]:
!python train.py "/content/drive/MyDrive/assignments/ITRex, men-women problem/training_data" "catboost.save"

Learning rate set to 0.139196
0:	learn: 0.6524512	test: 0.6486168	best: 0.6486168 (0)	total: 412ms	remaining: 2m 3s
50:	learn: 0.5286945	test: 0.5052711	best: 0.5052711 (50)	total: 17.4s	remaining: 1m 24s
100:	learn: 0.5215923	test: 0.5018273	best: 0.5018203 (99)	total: 43.5s	remaining: 1m 25s
150:	learn: 0.5139277	test: 0.5000768	best: 0.5000768 (150)	total: 1m 5s	remaining: 1m 5s
200:	learn: 0.5071080	test: 0.4990263	best: 0.4990078 (199)	total: 1m 30s	remaining: 44.4s
250:	learn: 0.5012409	test: 0.4987102	best: 0.4986313 (233)	total: 1m 53s	remaining: 22.1s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.4983307601
bestIteration = 276

Shrink model to first 277 iterations.
Classification report on validation dataset:               precision    recall  f1-score   support

           0       0.77      0.72      0.75      5937
           1       0.74      0.79      0.76      6063

    accuracy                           0.76     12000
   macro avg       0.76      0.7

In [40]:
!python classify.py "catboost.save" "This is a test string"

male
